### Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Change Directory
%cd '/content/drive/My Drive/dsp/project'

/content/drive/My Drive/dsp/project


###add tensorboard

In [4]:
pip install tensorboardX

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 11.0 MB/s eta 0:00:00


### Install Required Libraries


In [5]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import argparse
from tensorboardX import SummaryWriter
import torch.nn.functional as F
import time
import random
from sklearn import preprocessing
from scipy.signal import butter, lfilter


### Options

In [6]:
class Options:
    def __init__(self):
        pass

    def init(self, parser):
        # Global settings
        parser.add_argument('--batch_size', type=int, default=256,
                            help='Batch size for training and validation.')
        parser.add_argument('--nepoch', type=int, default=50,
                            help='Number of training epochs.')
        parser.add_argument('--lr_initial', type=float, default=0.001,
                            help='Initial learning rate for the optimizer.')
        parser.add_argument('--decay_epoch', type=int, default=20,
                            help='Epoch at which to start decaying the learning rate.')

        # Device settings
        parser.add_argument('--device', type=str, default='cuda',
                            help='Device to use for training ("cuda" for GPU, "cpu" for CPU).')

        # Model settings
        parser.add_argument('--classes', type=int, default=5,
                            help='Number of output classes for classification.')

        # Pretrained model settings
        parser.add_argument('--log_name', type=str, default='241212',
                            help='Identifier for logging and checkpointing.')
        parser.add_argument('--pretrained', type=bool, default=False,
                            help='Whether to load a pretrained model (True/False).')
        parser.add_argument('--pretrained_model', type=str,
                            default='./log/241212/models/ckpt_opt.pt',
                            help='Path to the pretrained model weights file.')

        # Dataset settings
        parser.add_argument('--fs', type=int, default=360,
                            help='Sampling frequency of the ECG data.')
        parser.add_argument('--path_train_data', type=str,
                            default='./dataset/train_data.npy',
                            help='Path to save the training data.')
        parser.add_argument('--path_train_labels', type=str,
                            default='./dataset/train_labels.npy',
                            help='Path to save the training labels.')
        parser.add_argument('--path_val_data', type=str,
                            default='./dataset/val_data.npy',
                            help='Path to save the validation data.')
        parser.add_argument('--path_val_labels', type=str,
                            default='./dataset/val_labels.npy',
                            help='Path to save the validation labels.')
        parser.add_argument('--path_test_data', type=str,
                            default='./dataset/test_data.npy',
                            help='Path to save the test data.')
        parser.add_argument('--path_test_labels', type=str,
                            default='./dataset/test_labels.npy',
                            help='Path to save the test labels.')


        return parser


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description='Options for ECG Classification Training')
    opt = Options().init(parser).parse_known_args()
    print(opt)


(Namespace(batch_size=256, nepoch=50, lr_initial=0.001, decay_epoch=20, device='cuda', classes=5, log_name='241212', pretrained=False, pretrained_model='./log/241212/models/ckpt_opt.pt', fs=360, path_train_data='./dataset/train_data.npy', path_train_labels='./dataset/train_labels.npy', path_val_data='./dataset/val_data.npy', path_val_labels='./dataset/val_labels.npy', path_test_data='./dataset/test_data.npy', path_test_labels='./dataset/test_labels.npy'), ['-f', '/root/.local/share/jupyter/runtime/kernel-7dd485ad-b82f-4768-9781-bda52dcf5da5.json'])


### Helper function

In [8]:
# For dataset
class ECGDataloader():  # 1110 - 4096 samples
    def __init__(self, data, label):
        self.data = data
        self.label = label

    def __getitem__(self, index):
        return (torch.tensor(self.data[index], dtype=torch.float32), torch.tensor(self.label[index], dtype=torch.long))

    def __len__(self):
        return len(self.data)


# Low-pass filter function
def lowpass_filter(data, cutoff = 30, fs = 360, order = 4):
    nyquist = 0.5 * fs
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype = 'low', analog = False)
    return lfilter(b, a, data)



# FIR filter
def preprocess_and_filter(self, data, cutoff = 30, fs = 360, numtaps = 101):
  # DC offset 제거
  data_centered = data - np.mean(data)

  #FIR filter 계수 계산
  nyquist = 0.5 * fs
  normal_cutoff = cutoff / nyquist
  fir_coeff = firwin(numtaps, normal_cutoff, pass_zero = "lowpass")

  #FIR filter 적용
  filtered_signal = lfilter(fir_coeff, 1.0, data_centered)

  # 지연보정
  delay = numtaps // 2
  filtered_signal = np.roll(filtered_signal, -delay)
  filtered_signal[-delay:] = 0 # 뒤쪽 데이터를 채움

  return filtered_signal

# For dataset
def label2index(i):
    m = {'N': 0, 'S': 1, 'V': 2, 'F': 3, 'Q': 4}  # uncomment for 5 classes
    return m[i]



# Create a new directory.
def mkdir(path):
    if not os.path.exists(path):
        os.makedirs(path)


# Normalize the ECG data using Z-score normalization.
def normalize_ecg(ecg_data):
    mean = np.mean(ecg_data, axis = 0, keepdims=True)
    std = np.std(ecg_data, axis = 0, keepdims=True)
    return (ecg_data - mean) / (std + 1e-8)  # Prevent division by zero


# for using pre-training weights
def optimizer_to(optim, device):
    for param in optim.state.values():
        # Not sure there are any global tensors in the state dict
        if isinstance(param, torch.Tensor):
            param.data = param.data.to(device)
            if param._grad is not None:
                param._grad.data = param._grad.data.to(device)
        elif isinstance(param, dict):
            for subparam in param.values():
                if isinstance(subparam, torch.Tensor):
                    subparam.data = subparam.data.to(device)
                    if subparam._grad is not None:
                        subparam._grad.data = subparam._grad.data.to(device)


# Calculate total number of parameters in a model.
def cal_total_params(our_model):
    total_parameters = 0
    for variable in our_model.parameters():
        shape = variable.size()
        variable_parameters = 1
        for dim in shape:
            variable_parameters *= dim
        total_parameters += variable_parameters

    return total_parameters


# Display a progress bar during training/validation.
class Bar(object):
    def __init__(self, dataloader):
        if not hasattr(dataloader, 'dataset'):
            raise ValueError('Attribute `dataset` not exists in dataloder.')
        if not hasattr(dataloader, 'batch_size'):
            raise ValueError('Attribute `batch_size` not exists in dataloder.')

        self.dataloader = dataloader
        self.iterator = iter(dataloader)
        self.dataset = dataloader.dataset
        self.batch_size = dataloader.batch_size
        self._idx = 0
        self._batch_idx = 0
        self._time = []
        self._DISPLAY_LENGTH = 50

    def __len__(self):
        return len(self.dataloader)

    def __iter__(self):
        return self

    def __next__(self):
        if len(self._time) < 2:
            self._time.append(time.time())

        self._batch_idx += self.batch_size
        if self._batch_idx > len(self.dataset):
            self._batch_idx = len(self.dataset)

        try:
            batch = next(self.iterator)
            self._display()
        except StopIteration:
            raise StopIteration()

        self._idx += 1
        if self._idx >= len(self.dataloader):
            self._reset()

        return batch

    def _display(self):
        if len(self._time) > 1:
            t = (self._time[-1] - self._time[-2])
            eta = t * (len(self.dataloader) - self._idx)
        else:
            eta = 0

        rate = self._idx / len(self.dataloader)
        len_bar = int(rate * self._DISPLAY_LENGTH)
        bar = ('=' * len_bar + '>').ljust(self._DISPLAY_LENGTH, '.')
        idx = str(self._batch_idx).rjust(len(str(len(self.dataset))), ' ')

        tmpl = '\r{}/{}: [{}] - ETA {:.1f}s'.format(
            idx,
            len(self.dataset),
            bar,
            eta
        )
        print(tmpl, end='')
        if self._batch_idx == len(self.dataset):
            print()

    def _reset(self):
        self._idx = 0
        self._batch_idx = 0
        self._time = []


# Define a custom writer class that extends SummaryWriter to log training/validation metrics.
class Writer(SummaryWriter):
    def __init__(self, logdir):
        super(Writer, self).__init__(logdir)

    # Method to log training loss.
    def log_train_loss(self, loss_type, train_loss, step):
        self.add_scalar('train_{}_loss'.format(loss_type), train_loss, step)

    # Method to log validation loss.
    def log_valid_loss(self, loss_type, valid_loss, step):
        self.add_scalar('valid_{}_loss'.format(loss_type), valid_loss, step)

    # Method to log other performance metrics (e.g., accuracy, F1-score).
    def log_score(self, metrics_name, metrics, step):
        # Add a scalar value to the writer with the given metric name.
        self.add_scalar(metrics_name, metrics, step)


def save_checkpoint(exp_log_dir, model, epoch):
    save_dict = {
        "model": model.state_dict(),
        'epoch': epoch
    }
    # save classification report
    save_path = os.path.join(exp_log_dir, "ckpt_opt.pt")

    torch.save(save_dict, save_path)


### ResNet Model

In [9]:
def conv_block(in_planes, out_planes, stride=1, groups=1, dilation=1):
    return nn.Conv1d(
        in_planes,
        out_planes,
        kernel_size=17,
        stride=stride,
        padding=8,
        groups=groups,
        bias=False,
        dilation=dilation,
    )


def conv_subsumpling(in_planes, out_planes, stride=1):
    return nn.Conv1d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)



class BasicBlock(nn.Module):
    expansion = 1

    def __init__(
        self,
        inplanes,
        planes,
        stride=1,
        downsample=None,
        groups=1,
        base_width=64,
        dilation=1,
        norm_layer=None,
    ):
        super(BasicBlock, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm1d
        if groups != 1 or base_width != 64:
            raise ValueError("BasicBlock only supports groups=1 and base_width=64")
        if dilation > 1:
            raise NotImplementedError("Dilation > 1 not supported in BasicBlock")
        # Both self.conv1 and self.downsample layers downsample the input when stride != 1
        self.conv1 = conv_block(inplanes, planes, stride)
        self.bn1 = norm_layer(inplanes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv_block(planes, planes)
        self.bn2 = norm_layer(planes)
        self.dropout = nn.Dropout()
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.bn1(x)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv1(out)

        out = self.bn2(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.conv2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity

        return out


class EcgResNet34(nn.Module):
    def __init__(
        self,
        opt,
        layers=(1, 5),
        num_classes=5,
        zero_init_residual=False,
        groups=1,
        width_per_group=64,
        replace_stride_with_dilation=None,
        norm_layer=None,
        block=BasicBlock,
    ):

        super(EcgResNet34, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm1d
        self._norm_layer = norm_layer

        self.inplanes = 32
        self.dilation = 1
        if replace_stride_with_dilation is None:
            # each element in the tuple indicates if we should replace
            # the 2x2 stride with a dilated convolution instead
            replace_stride_with_dilation = [False, False, False]
        if len(replace_stride_with_dilation) != 3:
            raise ValueError(
                "replace_stride_with_dilation should be None "
                "or a 3-element tuple, got {}".format(replace_stride_with_dilation),
            )
        self.groups = groups
        self.base_width = width_per_group
        self.conv1 = conv_block(1, self.inplanes, stride=1)
        self.bn1 = norm_layer(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        # 레이어를 2개만 설정하였음
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(
            block, 128, layers[1], stride=2, dilate=replace_stride_with_dilation[0],
        )
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(128 * block.expansion, num_classes)

        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, (nn.BatchNorm1d, nn.GroupNorm)):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

        # Zero-initialize the last BN in each residual branch,
        # so that the residual branch starts with zeros, and each residual block behaves like an identity.
        # This improves the model by 0.2~0.3% according to https://arxiv.org/abs/1706.02677
        if zero_init_residual:
            for m in self.modules():
                if isinstance(m, BasicBlock):
                    nn.init.constant_(m.bn2.weight, 0)

    def _make_layer(self, block, planes, blocks, stride=1, dilate=False):
        norm_layer = self._norm_layer
        downsample = None
        previous_dilation = self.dilation
        if dilate:
            self.dilation *= stride
            stride = 1
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv_subsumpling(self.inplanes, planes * block.expansion, stride),
                norm_layer(planes * block.expansion),
            )

        layers = []
        layers.append(
            block(
                self.inplanes,
                planes,
                stride,
                downsample,
                self.groups,
                self.base_width,
                previous_dilation,
                norm_layer,
            ),
        )
        self.inplanes = planes * block.expansion
        for _ in range(1, blocks):
            layers.append(
                block(
                    self.inplanes,
                    planes,
                    groups=self.groups,
                    base_width=self.base_width,
                    dilation=self.dilation,
                    norm_layer=norm_layer,
                ),
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)

        x = self.layer1(x)
        x = self.layer2(x)

        x = self.avgpool(x)
        x = x.reshape(x.size(0), -1)
        x = self.fc(x)

        return x

### Training

In [ ]:
class Trainer:
    def __init__(self, opt):
        self.opt = opt
        self.label_encoder = LabelEncoder()
        self.model = EcgResNet34(opt).to(opt.device)
        self.loss_fn = torch.nn.CrossEntropyLoss()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=opt.lr_initial)
        self.scheduler = torch.optim.lr_scheduler.StepLR(self.optimizer, step_size=opt.decay_epoch, gamma=0.1)
        self.writer = Writer(self._get_tboard_dir())
        self.train_loader, self.valid_loader = self._load_data()
        self.log_file_path = os.path.join(self._get_tboard_dir(), 'training_log.txt')


        # Load pretrained model if specified
        if self.opt.pretrained:
            self._load_pretrained_model()

    # Z-score normalization
    def _normalize_ecg(self, ecg_data, axis=1):
        mean = np.mean(ecg_data, axis=axis, keepdims=True)
        std = np.std(ecg_data, axis=axis, keepdims=True)
        return (ecg_data - mean) / (std + 1e-8)  # Prevent division by zero

    def _load_data(self):
        # Load and preprocess data
        train_data = np.load(self.opt.path_train_data, allow_pickle=True)  # Load ECG train data
        train_labels = np.load(self.opt.path_train_labels, allow_pickle=True)  # Load train label
        val_data = np.load(self.opt.path_val_data, allow_pickle=True)  # Load ECG validation data
        val_labels = np.load(self.opt.path_val_labels, allow_pickle=True)  # Load validation labels

        Y_train = np.array([label2index(i) for i in train_labels])  # Convert labels to indices
        Y_val = np.array([label2index(i) for i in val_labels])  # Convert labels to indices

        # Low pass filter
        #train_data = utils.lowpass_filter(train_data, cutoff=30, fs=360)

        # Normalize data along the time axis
        # 조금 수정하였음
        X_train = self._normalize_ecg(train_data)
        X_val = self._normalize_ecg(val_data)

        # Expand dimensions to match model input requirements
        # 기존 명령어
        X_train = np.expand_dims(X_train, 1)
        X_val = np.expand_dims(X_val, 1)

        # 추가된 부분
        # X_train = X_train[:, None, :, None]  # Shape: (samples, channels, height, width)
        #X_val = X_val[:, None, :, None]

        Y_train = self.label_encoder.fit_transform(Y_train)
        Y_val = self.label_encoder.transform(Y_val)



        train_dataset = ECGDataloader(X_train, Y_train)     # X_train : train_data, Y_train : train labels
        val_dataset = ECGDataloader(X_val, Y_val)
        # Create DataLoader for training and validation
        train_loader = DataLoader(train_dataset, batch_size=self.opt.batch_size, shuffle=True,
                                  num_workers=2)#, pin_memory=True, drop_last=True)
        valid_loader = DataLoader(val_dataset, batch_size=self.opt.batch_size, shuffle=False,
                                  num_workers=2)

        return train_loader, valid_loader

    def _get_tboard_dir(self):
        # Initialize directories for logging and model storage
        log_dir = os.path.join(os.getcwd(), 'log', f'{self.opt.log_name}')
        mkdir(log_dir)
        mkdir(os.path.join(log_dir, 'logs'))
        mkdir(os.path.join(log_dir, 'models'))
        return os.path.join(log_dir, 'logs')

    def _load_pretrained_model(self):
        # Load pretrained model weights if specified
        print('Loading the pretrained model...')
        chkpt = torch.load(self.opt.pretrained_model)
        self.model.load_state_dict(chkpt['model'])
        self.optimizer.load_state_dict(chkpt['optimizer'])
        optimizer_to(self.optimizer, self.opt.device)
        print('Resuming Start Epoch:', chkpt['epoch'] + 1)

    def train(self):
        # Print the total number of parameters in the model
        print(
            f'Total parameters: {cal_total_params(self.model):,} ({cal_total_params(self.model) / 1e6:.2f}M)')
        best_f1 = 0
        for epoch in range(1, self.opt.nepoch + 1):
            start_time = time.time()
            self.model.train()
            train_loss = 0

            # Training loop
            for X, Y in Bar(self.train_loader):
                X, Y = X.float().to(self.opt.device), Y.long().to(self.opt.device)  # Move data to device

                # Forward pass and optimization
                outputs = self.model(X)  # Get model predictions
                loss = self.loss_fn(outputs, Y)  # Calculate loss
                self.optimizer.zero_grad()  # Clear previous gradients
                loss.backward()  # Backpropagate to calculate gradients
                self.optimizer.step()  # Update model parameters
                train_loss += loss.item()

            avg_train_loss = train_loss / len(self.train_loader)  # Calculate average training loss
            self.writer.log_train_loss('total', avg_train_loss, epoch)  # Log training loss

            # Validation
            accuracy, f1, avg_val_loss = self._evaluate(self.valid_loader, epoch)  # Evaluate model on validation set
            if f1 > best_f1:  # Save the best model based on accuracy
                best_f1 = f1
                save_checkpoint(self._get_model_dir(), self.model, epoch)

            self.writer.log_score('F1-score', f1, epoch)  # Log validation accuracy
            self.scheduler.step()  # Update learning rate scheduler

            # Logging
            log_message = (
                f'EPOCH[{epoch}] Train Loss: {avg_train_loss:.6f} | Validation Loss: {avg_val_loss:.6f} | Validation F1-score: {f1:.6f} | Time: {time.time() - start_time:.3f}s'
            )
            print(log_message)
            self._log_to_file(log_message)

        print('Training completed.')

    def _evaluate(self, dataloader, epoch):
        # Evaluate the model on the given dataloader
        self.model.eval()
        pred_labels, true_labels = [], []
        total_loss = 0
        with torch.no_grad():
            for X, Y in Bar(dataloader):
                X, Y = X.float().to(self.opt.device), Y.long().to(self.opt.device)  # Move data to device
                pred = self.model(X)  # Get model predictions
                loss = self.loss_fn(pred, Y)  # Calculate loss
                total_loss += loss.item()

                # Get predicted class directly from raw logits
                pred_classes = torch.argmax(pred, dim=1)
                pred_labels.extend(pred_classes.cpu().numpy())
                true_labels.extend(Y.cpu().numpy())

        # Calculate accuracy
        pred_labels = np.array(pred_labels)
        true_labels = np.array(true_labels)
        accuracy = np.mean(pred_labels == true_labels)  # Calculate accuracy
        avg_valid_loss = total_loss / len(dataloader)  # Calculate average validation loss
        f1 = f1_score(true_labels, pred_labels, average='weighted')  # Calculate F1-score
        self.writer.log_valid_loss('total', avg_valid_loss, epoch)  # Log validation loss
        return accuracy, f1, avg_valid_loss

    def _get_model_dir(self):
        # Get directory path for saving models
        log_dir = os.path.join(os.getcwd(), 'log', f'{self.opt.log_name}')
        return os.path.join(log_dir, 'models')

    def _log_to_file(self, message):
        # Write log message to file
        with open(self.log_file_path, 'a') as f:
            f.write(message + '\n')



if __name__ == '__main__':
    # Parse command-line arguments
    opt = Options().init(argparse.ArgumentParser(description='ECG Classification')).parse_known_args()
    print(opt[0])

    # Set random seeds for reproducibility
    torch.manual_seed(1234)
    np.random.seed(1234)
    random.seed(1234)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(1234)

    # Initialize trainer and start training
    trainer = Trainer(opt[0])
    trainer.train()


Namespace(batch_size=256, nepoch=50, lr_initial=0.001, decay_epoch=20, device='cuda', classes=5, log_name='241210_99', pretrained=False, pretrained_model='./log/241210_99/models/ckpt_opt.pt', fs=360, path_train_data='./dataset/train_data.npy', path_train_labels='./dataset/train_labels.npy', path_val_data='./dataset/val_data.npy', path_val_labels='./dataset/val_labels.npy', path_test_data='./dataset/test_data.npy', path_test_labels='./dataset/test_labels.npy')
Total parameters: 2,764,965 (2.76M)
63015/63015: [=================================================>] - ETA 1.8s
21005/21005: [=================================================>] - ETA 0.1s
EPOCH[1] Train Loss: 0.404560 | Validation Loss: 0.207022 | Validation F1-score: 0.930198 | Time: 46.807s
63015/63015: [=================================================>] - ETA 0.2s
21005/21005: [=================================================>] - ETA 0.1s
EPOCH[2] Train Loss: 0.170864 | Validation Loss: 0.121501 | Validation F1-score: 0.963

### Testing

In [ ]:
class Tester:
    def __init__(self, opt):
        self.opt = opt
        self.label_encoder = LabelEncoder()
        self.model = EcgResNet34(opt).to(opt.device)
        self._load_pretrained_model()
        self.test_loader = self._load_data()

    # Z-score normalization
    def _normalize_ecg(self, ecg_data, axis=1):
        mean = np.mean(ecg_data, axis=axis, keepdims=True)
        std = np.std(ecg_data, axis=axis, keepdims=True)
        return (ecg_data - mean) / (std + 1e-8)  # Prevent division by zero

    def _load_data(self):
        # Load and preprocess data
        test_data = np.load(self.opt.path_val_data)  # Load ECG train data
        test_labels = np.load(self.opt.path_val_labels)  # Load train labels
        X_test = self._normalize_ecg(test_data)
        Y_test = np.array([label2index(i) for i in test_labels])

        # Prepare DataLoader
        #X_test = test_data[:, None, :, None]
        # 수정된 부분
        X_test = np.expand_dims(test_data, 1)
        Y_test = self.label_encoder.fit_transform(Y_test)
        #


        val_dataset = ECGDataloader(X_test, Y_test)
        test_loader = DataLoader(val_dataset, batch_size=self.opt.batch_size, shuffle=False, num_workers=2)
        return test_loader

    def _load_pretrained_model(self):
        # Load the pretrained model for evaluation
        print('Loading the pretrained model...')
        chkpt = torch.load(self.opt.pretrained_model, map_location=self.opt.device)
        self.model.load_state_dict(chkpt['model'])

    def test(self):
        self.model.eval()
        pred_labels, true_labels = [], []
        total_loss = 0
        loss_fn = torch.nn.CrossEntropyLoss()

        with torch.no_grad():
            for X, Y in Bar(self.test_loader):
                X, Y = X.float().to(self.opt.device), Y.long().to(self.opt.device)
                outputs = self.model(X)
                loss = loss_fn(outputs, Y)
                total_loss += loss.item()

                # Get predicted classes
                pred_classes = torch.argmax(outputs, dim=1)
                pred_labels.extend(pred_classes.cpu().numpy())
                true_labels.extend(Y.cpu().numpy())

        # Calculate accuracy
        pred_labels = np.array(pred_labels)
        true_labels = np.array(true_labels)
        accuracy = np.mean(pred_labels == true_labels)
        avg_test_loss = total_loss / len(self.test_loader)
        f1 = f1_score(true_labels, pred_labels, average='weighted')  # Calculate F1-score

        # Print results in a more professional format
        print(f'==================== Test Results ====================')
        # print(f'| Test Accuracy    : {accuracy * 100:.2f}%')
        print(f'| Test F1-score    : {f1 * 100:.2f}%')
        print(f'=======================================================')

if __name__ == '__main__':
    opt = Options().init(argparse.ArgumentParser(description='ECG Classification Testing')).parse_known_args()
    torch.manual_seed(1234)
    np.random.seed(1234)
    random.seed(1234)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(1234)

    tester = Tester(opt[0])
    tester.test()



Loading the pretrained model...


<ipython-input-9-9ff6a95907a9>:37: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  chkpt = torch.load(self.opt.pretrained_model, map_location=self.opt.device)


21005/21005: [=================================================>] - ETA 1.1s
==================== Test Results ====================
| Test F1-score    : 99.29%
